# Unit 3 — PPO: Proximal Policy Optimisation

> *Part of the [RL for Robotics & LLMs](https://github.com/AliBuildsAI/rl-for-robotics-llms) series.*

In Unit 2, A2C + GAE gave us a much better advantage estimator and stable
N-step rollouts — but it still has one fundamental limitation: **one gradient
update per rollout, with no constraint on step size**.  A single bad minibatch
can still overwrite a good policy.

This unit fixes that with **Proximal Policy Optimisation (PPO, Schulman et al.
2017)**, which lets us reuse each rollout for multiple gradient steps safely —
by clipping the probability ratio so the policy can never move too far from
where the data was collected.

We demonstrate on **LunarLander-v3**: A2C plateaus well below the solved
threshold; PPO reliably crosses it.

**What we build:**

| Part | Algorithm | Key change over previous unit |
|------|-----------|-------------------------------|
| A | A2C + GAE (Unit 2 baseline) | One gradient step per rollout |
| B | PPO-Clip | $K$ epochs × minibatches · clipped surrogate loss |


In [ ]:
%pip install -q swig
%pip install -q "gymnasium[box2d]" torch imageio pillow matplotlib
print("All packages ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
from IPython.display import Image, display

print(f"gymnasium {gym.__version__}  |  torch {torch.__version__}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


---
## 1  Environment — LunarLander-v3

Before the algorithm, we need to understand the problem precisely.  This
section is a from-scratch tour of LunarLander: what the task is, what each
action does *physically*, and how every term of the reward steers the agent.

### What is a lander?

A **lander** is a spacecraft built to set down on a planet or moon.  It flies
through space, then slowly descends and touches the surface.  LunarLander
simulates a small craft trying to land without crashing — think of it as a
very careful *vertical* parking job, falling out of the sky.

### Gravity

**Gravity** is the invisible force pulling everything toward the ground.  Drop
a ball and it falls because Earth pulls it toward its centre.  The Moon has
gravity too, but ~6× weaker — things fall more slowly there.  In
**LunarLander-v3** the default gravity is about $10\,\text{m/s}^2$ downward
(close to Earth, not the real Moon — "Lunar" is just the theme; the physics
constants are configurable).  Either way, the craft is *always* being pulled
down.

### Thrust

**Thrust** is the push an engine produces, and it is pure Newton's 3rd law —
"every action has an equal and opposite reaction."  A rocket throws gas fast in
one direction; the reaction pushes the rocket the other way.  Stand on ice
holding a heavy ball: throw it right, you slide left.  That leftward push is
thrust.  In LunarLander:

- the **main engine** throws gas *downward* → the craft is pushed *upward*,
  fighting gravity;
- the **side engines** throw gas *sideways* → the craft is pushed sideways,
  steering it.

With the engine off, only gravity acts and the craft accelerates downward
faster and faster — hit the ground fast and it crashes.  The agent's whole job
is to fire the engines at the right moments to arrive gently and in the right
place.

```
                 ↑ thrust (main engine fighting gravity)
              ___|___
             |  🚀   |   ← the lander (spacecraft)
             |______|
                 |
                 ↓ gravity (always pulling down)

    ___________🏳️___🏳️____________
               landing pad
               (goal: land here)
```


### The actual objective

Land the craft **softly on the pad** — the flat strip between the two flags,
centred at $x = 0$ — without crashing.  Concretely:

- **Position:** directly over the pad ($x \approx 0$, $y \approx 0$)
- **Velocity:** nearly zero (a soft touchdown, not a slam)
- **Orientation:** upright ($\theta \approx 0$)
- **Legs:** both contact the ground
- **Fuel:** use as little as possible

### State — what the agent observes

8 continuous values:

- position $(x, y)$
- velocity $(\dot{x}, \dot{y})$
- angle $\theta$ and angular velocity $\dot{\theta}$
- two leg-contact booleans $c_L, c_R$

**Solved:** average reward $\geq 200$ over 100 consecutive greedy episodes.


### Actions — what each one does physically

There are 4 discrete actions.  The key trap: an engine's name describes *where
it sits*, not the direction it pushes.

```
         main engine (action 2)
              ↑  thrust
         ___↑___
        |       |
        |  body |
     ↙  |       |  ↘
left    ───────   right
engine              engine
(action 1)        (action 3)
```

**Action 0 — Do nothing.** No thrust. Gravity keeps accelerating the craft
downward at ~$10\,\text{m/s}^2$; any existing velocity and spin continue
unchanged. Costs no fuel. The agent uses this when it is already falling the
right way and needs no correction.

**Action 2 — Main engine** (bottom-centre). Fires downward, so by Newton's 3rd
law the craft is pushed **upward** — the only way to slow vertical descent. The
most powerful action (`MAIN_ENGINE_POWER = 13`) and the most expensive
(`−0.30` fuel/step). Fire too little → crash; fire too much → waste fuel and
overshoot.

**Action 1 — Left orientation engine.** The engine on the **left side** fires;
exhaust goes left, so the reaction pushes the craft **right**. Applied
off-centre, it also creates a torque that rotates the craft **clockwise** (top
tilts right, $\theta$ increases). Net: drift right + tilt clockwise. Fuel cost
`−0.03` (10× cheaper than the main engine).

**Action 3 — Right orientation engine.** Mirror image: exhaust goes right,
craft pushed **left**, rotates **counter-clockwise** ($\theta$ decreases). Net:
drift left + tilt counter-clockwise. Fuel cost `−0.03`.

> **The inversion the agent must learn:** "left engine" pushes you *right*. To
> correct rightward drift you fire the *right* engine (action 3). The names
> refer to engine *position*, not thrust direction.


### The reward — how each term guides the agent

Most of the per-step reward is the **change in a shaping potential** $\Phi$.
First the potential itself (higher = closer to a good landing state):

$$\Phi_t = -100\sqrt{x^2 + y^2}\; -\; 100\sqrt{\dot{x}^2 + \dot{y}^2}\; -\; 100\,|\theta|\; +\; 10\,c_L + 10\,c_R$$

with terms **(1)** distance, **(2)** speed, **(3)** tilt, **(4)** leg contact.
The reward at each step is the *change* in $\Phi$, plus a fuel cost:

$$r_t = \Phi_t - \Phi_{t-1}\; -\; 0.30\cdot\mathbf{1}[\text{main}]\; -\; 0.03\cdot\mathbf{1}[\text{side}]$$

**Why the change $\Phi_t - \Phi_{t-1}$ and not $\Phi_t$ directly?** With
$\Phi_t$ directly, the agent would be punished just for *starting* far from the
pad, no matter what it does. Using the change means it only earns a positive
signal when $\Phi$ **improved** this step: move toward the pad → $\Phi$ rises →
positive reward; move away → negative. Every single step carries meaningful
feedback. (This is *potential-based shaping* — it changes the per-step signal
without changing which policy is optimal.)

**Term 1 — Distance** $-100\sqrt{x^2+y^2}$: navigate to the right place. The
craft starts around $(0, 1.4)$ — above the pad but high. Every tiny drift
toward $(0,0)$ shows up as reward next step.

**Term 2 — Speed** $-100\sqrt{\dot{x}^2+\dot{y}^2}$: arrive slowly. Being over
the pad is worthless at 10 m/s — that's a crash. This penalises kinetic energy
and conflicts productively with term 1: fall toward the pad, but brake on the
way. That tension *is* "soft landing."

**Term 3 — Tilt** $-100\,|\theta|$: stay upright so the legs touch first. Tilt
too far and the *body* hits — instant crash. Staying vertical also keeps the
main engine's thrust pointing straight down instead of wasting it sideways.

**Term 4 — Leg contact** $+10\,c_L + 10\,c_R$: a leg touching down jumps $\Phi$
by $+10$ → big positive $\Delta\Phi$; a leg *losing* contact (a bounce) drops it
by $10$ → penalty. This rewards a stable final touchdown and discourages
endless hovering just above the ground.

**Term 5 — Fuel** $-0.30$ (main) / $-0.03$ (side): the main engine costs 10× a
side engine because it is 10× more powerful. Without this, the agent would
hover forever — cancelling gravity with the main engine satisfies position,
speed and tilt yet never actually lands. The fuel cost makes hovering expensive
and forces a decisive descent.

**Terminal events:**

$$r_T = +100 \quad\text{if the craft settles (both legs down, near-zero velocity)}$$
$$r_T = -100 \quad\text{if the body hits the ground (crash) or } |x| \geq 1 \text{ (off screen)}$$

These are the real win/lose conditions; everything above is dense shaping that
guides the agent toward the $+100$ and away from the $-100$.

**Why all five terms are necessary** — remove any one and something breaks:

| If you remove… | What happens |
|---|---|
| Distance | No incentive to go anywhere specific — hovers randomly |
| Speed | Crashes at full speed — reaches the pad but never brakes |
| Tilt | Tips sideways, body hits first, crash |
| Leg contact | Hovers just above the pad forever, never lands |
| Fuel cost | Hovers indefinitely at the pad, never commits to touchdown |

Together they form the recipe: navigate → brake → stay upright → touch down → stop.


### Why this environment for PPO?

Published SB3 benchmarks show A2C + GAE on LunarLander only reaches roughly
~30 average reward even when well tuned — and in **Part A** below our own A2C
baseline plateaus around $-147$, far short of the 200 threshold.  The
environment is hard enough that A2C's single-update-per-rollout regime can't
converge reliably.  **PPO needs to be the hero here**, and Part B shows it
crossing 200.


In [ ]:
env_peek = gym.make("LunarLander-v3")
STATE_DIM  = env_peek.observation_space.shape[0]   # 8
ACTION_DIM = env_peek.action_space.n               # 4
print(f"State dim : {STATE_DIM}")
print(f"Action dim: {ACTION_DIM}")
env_peek.close()


---
## 2  Why A2C Still Fails — The Step-Size Problem

A2C + GAE gave us two things:

1. **Better advantage estimates** via GAE (less variance)
2. **Stable gradient scale** via N-step rollouts from parallel envs

What it did NOT give us: any constraint on **how large a single gradient step
can be**.

After collecting a rollout, A2C does exactly one Adam step.  If that step is
large — because the advantages happened to be high, or the gradient is
pointing in a slightly wrong direction — the policy can jump to a bad region
of parameter space and collapse.

The natural fix: **reuse each rollout for $K$ gradient steps** instead of one.
More gradient steps per batch of data = better sample efficiency.

But here is the problem with naively reusing data.


---
## 3  Why Naive Multi-Step Reuse Fails — Distribution Shift

The policy gradient estimator is:

$$\hat{g} = \frac{1}{|D|}\sum_{\tau \in D} \sum_t
\nabla_\theta \log \pi_\theta(a_t|s_t)\,{\hat{A}}_t$$

This is an unbiased estimate of $\nabla_\theta J(\theta)$ **only when the
data $D$ was collected under the current policy $\pi_\theta$**.

After the first gradient step, $\theta$ has changed.  The data was collected
under $\pi_{\theta_{\text{old}}}$, not $\pi_\theta$.  Every subsequent step
uses data from the wrong distribution.  The gradient is biased, and the policy
degrades.

The solution: **importance sampling** — a mathematical correction that lets
you evaluate gradients under the new policy using data from the old one.


---
## 4  Importance Sampling — Exact Correction for Off-Distribution Data

**The IS identity.** For any two distributions $p$ and $q$ with $p$ absolutely
continuous with respect to $q$:

$$\mathbb{E}_{x \sim p}[f(x)] = \mathbb{E}_{x \sim q}\!\left[
  \frac{p(x)}{q(x)}\,f(x)
\right]$$

*Proof sketch:* write the left side as an integral, multiply and divide by $q(x)$:

$$\int f(x)\,p(x)\,dx = \int f(x)\,\frac{p(x)}{q(x)}\,q(x)\,dx
= \mathbb{E}_{x \sim q}\!\left[\frac{p(x)}{q(x)}\,f(x)\right]$$

The ratio $p(x)/q(x)$ is the **importance weight** — it reweights samples from
$q$ to produce correct expectations under $p$.

### Applied to trajectories

The objective is:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_t r_t\right]
= \mathbb{E}_{\tau \sim \pi_{\theta_{\text{old}}}}\!\left[
  \frac{p_\theta(\tau)}{p_{\theta_{\text{old}}}(\tau)}\sum_t r_t
\right]$$

The trajectory probability is:

$$p_\theta(\tau) = p(s_0) \prod_t p(s_{t+1}|s_t,a_t) \cdot \pi_\theta(a_t|s_t)$$

where $p(s_{t+1}|s_t,a_t)$ are the **environment dynamics** — identical in
numerator and denominator, independent of $\theta$, so they cancel:

$$\frac{p_\theta(\tau)}{p_{\theta_{\text{old}}}(\tau)}
= \prod_t \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

### The surrogate loss

Using the per-step advantage decomposition and the IS correction, the
**surrogate objective** to maximise is:

$$L^{\text{IS}}(\theta) = \mathbb{E}_t\!\left[
  \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}\,\hat{A}_t
\right] = \mathbb{E}_t\!\left[r_t(\theta)\,\hat{A}_t\right]$$

where $r_t(\theta) = \pi_\theta(a_t|s_t) \;/\; \pi_{\theta_{\text{old}}}(a_t|s_t)$
is the **probability ratio**.

Note that at $\theta = \theta_{\text{old}}$: $r_t = 1$ and we recover
the standard policy gradient — vanilla PG is just IS evaluated at the
collection point.

### Why you still need a trust region

IS is mathematically exact, but the IS estimator has high variance when
$\pi_\theta$ drifts far from $\pi_{\theta_{\text{old}}}$ — the ratio $r_t$
becomes large, amplifying any noise in $\hat{A}_t$.  More importantly, the
first-order surrogate is only a valid approximation of $J(\theta)$ near
$\theta_{\text{old}}$.  Taking large steps makes the approximation inaccurate.

**Constraining $r_t$ to stay near 1** bounds how far $\pi_\theta$ can deviate
from $\pi_{\theta_{\text{old}}}$ — which is exactly what PPO does.


---
## 5  PPO-Clip — A Structural Trust Region

TRPO enforces a hard KL constraint via a constrained optimisation solve
(natural gradient, conjugate gradient, line search) — correct but expensive.

PPO-Clip achieves a similar effect with a single line of code by **clipping**
the probability ratio to $[1-\varepsilon,\,1+\varepsilon]$:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[
  \min\!\left(
    r_t(\theta)\,\hat{A}_t,\;
    \text{clip}\bigl(r_t(\theta),\,1-\varepsilon,\,1+\varepsilon\bigr)\,\hat{A}_t
  \right)
\right]$$

The $\min$ makes this a **pessimistic lower bound** — we take the worse of
the two estimates.  The clip then kills the gradient whenever the policy
tries to move outside the trust region.

### The 4 cases

Fix a timestep $t$ and consider all combinations of advantage sign and ratio
position:

| | $r_t < 1-\varepsilon$ (ratio too low) | $1-\varepsilon \leq r_t \leq 1+\varepsilon$ | $r_t > 1+\varepsilon$ (ratio too high) |
|--|--|--|--|
| $\hat{A}_t > 0$ (good action) | clipped at $1-\varepsilon$ — **no gradient** | unclipped — gradient flows | clipped at $1+\varepsilon$ — **no gradient** |
| $\hat{A}_t < 0$ (bad action) | clipped at $1-\varepsilon$ — **no gradient** | unclipped — gradient flows | clipped at $1+\varepsilon$ — **no gradient** |

Gradient only flows when $r_t \in [1-\varepsilon,\,1+\varepsilon]$ —
**inside the trust region**.  Outside, the clip activates and the update
stops automatically.  No KL computation, no Lagrange multiplier, no line
search.

### Why $\varepsilon = 0.2$ works universally

The clip defines a symmetric 20% band around the old policy.  Empirically
this is wide enough for meaningful updates but tight enough to prevent
collapse.  Unlike TRPO's KL threshold, $\varepsilon$ has an intuitive
interpretation (allowed percentage change in action probabilities) and does
not need per-environment tuning.

### Connection to PPO-Lagrangian (PPO v1)

An equivalent formulation penalises the KL directly:

$$L^{\text{KLPEN}}(\theta) = \mathbb{E}_t[r_t(\theta)\hat{A}_t] - \beta\,
\text{KL}[\pi_{\theta_{\text{old}}}\|\pi_\theta]$$

where $\beta$ is a Lagrange multiplier that self-adjusts: if the observed KL
exceeds the target, $\beta$ increases; if below, $\beta$ decreases.
PPO-Clip is simpler and works at least as well in practice.


---
## 6  Networks

Same architecture and initialisation as Unit 2.
LunarLander's 8-dimensional state needs wider networks than Acrobot (256 vs 64).


In [ ]:
def layer_init(layer: nn.Linear, std: float = np.sqrt(2)) -> nn.Linear:
    """Orthogonal init — standard for PPO (CleanRL, SB3, OpenAI baselines)."""
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, 0.0)
    return layer


class ActorNetwork(nn.Module):
    """
    State → action logits.
    Input:  (state_dim,)   = (8,)  for LunarLander
    Output: (action_dim,)  = (4,)  raw logits → Categorical(logits=...)
    """
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            layer_init(nn.Linear(state_dim, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, action_dim), std=0.01),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def get_action(self, state: torch.Tensor, action: torch.Tensor = None):
        dist = torch.distributions.Categorical(logits=self.forward(state))
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), dist.entropy()


class CriticNetwork(nn.Module):
    """State → scalar V(s)."""
    def __init__(self, state_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            layer_init(nn.Linear(state_dim, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, 1), std=1.0),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


---
## 7  GAE — Unchanged from Unit 2

PPO uses exactly the same GAE computation as A2C.
The advantage estimator is not what changes — only the loss function does.


In [ ]:
def compute_gae(
    rewards:     torch.Tensor,   # (T, N_envs)
    values:      torch.Tensor,   # (T, N_envs) — detached
    dones:       torch.Tensor,   # (T, N_envs)
    next_values: torch.Tensor,   # (N_envs,)   — bootstrap
    gamma: float = 0.999,
    lam:   float = 0.98,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Backward GAE recurrence with episode-boundary masking.
    Returns (advantages, returns) both flattened to (T*N_envs,).
    Advantages are normalised; returns are raw (critic regression targets).
    """
    T, N = rewards.shape
    advantages = torch.zeros_like(rewards)
    gae = torch.zeros(N, device=rewards.device)

    for t in reversed(range(T)):
        nonterminal = 1.0 - dones[t]
        next_v = values[t + 1] if t < T - 1 else next_values
        delta = rewards[t] + gamma * next_v * nonterminal - values[t]
        gae = delta + gamma * lam * nonterminal * gae
        advantages[t] = gae

    returns  = advantages + values
    adv_flat = advantages.reshape(-1)
    ret_flat = returns.reshape(-1)
    adv_flat = (adv_flat - adv_flat.mean()) / (adv_flat.std() + 1e-8)
    return adv_flat, ret_flat


---
## Part A — A2C + GAE Baseline on LunarLander

The Unit 2 algorithm: one gradient update per rollout.

Each algorithm runs with **its own tuned hyperparameters** from
[rl-baselines3-zoo](https://github.com/DLR-RM/rl-baselines3-zoo):

| | A2C | PPO |
|--|-----|-----|
| `n_steps` | 5 | 1024 |
| `gamma` | 0.995 | 0.999 |
| `gae_lambda` | 0.95 | 0.98 |
| `n_epochs` | 1 | 4 |
| `minibatch_size` | — (full batch) | 64 |

A2C's small `n_steps=5` means 12,500 gradient updates in 500k steps —
much more frequent than PPO's 61 rollouts in 500k steps (but PPO gets
512 gradient steps per rollout vs A2C's 1).

Expected: A2C plateaus well below 200.
SB3's pre-trained A2C on LunarLander-v3 achieves ~30 average reward
even with optimal tuning — the single-update bottleneck is the issue,
not the hyperparameters.


In [ ]:
def a2c_gae(
    actor, critic, actor_opt, critic_opt,
    n_envs:       int   = 8,
    n_steps:      int   = 5,       # SB3-Zoo tuned for A2C on LunarLander
    total_steps:  int   = 500_000,
    gamma:        float = 0.995,   # SB3-Zoo tuned for A2C on LunarLander
    lam:          float = 0.95,
    entropy_coef: float = 0.01,
    value_coef:   float = 0.5,
    max_grad_norm:float = 0.5,
    print_every:  int   = 100_000,
    seed:         int   = 42,
) -> tuple[list, list]:
    """A2C + GAE — Unit 2 algorithm, one update per rollout."""
    envs = gym.vector.SyncVectorEnv(
        [lambda: gym.make("LunarLander-v3") for _ in range(n_envs)]
    )
    obs, _ = envs.reset(seed=seed)
    obs = torch.as_tensor(obs, dtype=torch.float32, device=device)

    ep_buf = np.zeros(n_envs)
    all_ep_rew, step_at_ep = [], []
    n_updates = total_steps // (n_steps * n_envs)

    for update in range(n_updates):
        steps_so_far = update * n_steps * n_envs
        obs_b = torch.zeros(n_steps, n_envs, STATE_DIM, device=device)
        act_b = torch.zeros(n_steps, n_envs, dtype=torch.long, device=device)
        rew_b = torch.zeros(n_steps, n_envs, device=device)
        don_b = torch.zeros(n_steps, n_envs, device=device)
        val_b = torch.zeros(n_steps, n_envs, device=device)

        for step in range(n_steps):
            with torch.no_grad():
                action, _, _ = actor.get_action(obs)
                value = critic(obs)
            next_obs, reward, terminated, truncated, _ = envs.step(action.cpu().numpy())
            done = (terminated | truncated).astype(np.float32)
            ep_buf += reward
            for i, d in enumerate(done):
                if d:
                    all_ep_rew.append(float(ep_buf[i]))
                    step_at_ep.append(steps_so_far + (step + 1) * n_envs)
                    ep_buf[i] = 0.0
            obs_b[step] = obs; act_b[step] = action
            rew_b[step] = torch.as_tensor(reward, dtype=torch.float32, device=device)
            don_b[step] = torch.as_tensor(done, dtype=torch.float32, device=device)
            val_b[step] = value
            obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)

        with torch.no_grad():
            next_val = critic(obs)

        advantages, returns = compute_gae(rew_b, val_b.detach(), don_b, next_val, gamma, lam)
        obs_flat = obs_b.reshape(-1, STATE_DIM)
        act_flat = act_b.reshape(-1)
        _, new_log_probs, entropy = actor.get_action(obs_flat, act_flat)
        new_values = critic(obs_flat)

        actor_loss  = -(new_log_probs * advantages).mean()
        critic_loss = value_coef * F.mse_loss(new_values, returns.detach())
        entropy_loss = -entropy_coef * entropy.mean()

        actor_opt.zero_grad(); critic_opt.zero_grad()
        (actor_loss + critic_loss + entropy_loss).backward()
        nn.utils.clip_grad_norm_(list(actor.parameters()) + list(critic.parameters()), max_grad_norm)
        actor_opt.step(); critic_opt.step()

        steps_done = (update + 1) * n_steps * n_envs
        if steps_done // print_every > (steps_done - n_steps * n_envs) // print_every and all_ep_rew:
            avg = np.mean(all_ep_rew[-100:]) if len(all_ep_rew) >= 100 else np.mean(all_ep_rew)
            print(f"[A2C]  step {steps_done:7d}  eps={len(all_ep_rew):4d}  avg100: {avg:7.1f}")

    envs.close()
    return all_ep_rew, step_at_ep


---
## Part B — PPO-Clip

Three changes from A2C, in order of importance:

1. **$K$ epochs over the same rollout** — instead of one gradient step, we
   loop over the rollout $K=4$ times with minibatches, giving many gradient
   steps per rollout.

2. **Long rollouts + small minibatches** — `n_steps=1024` means each rollout
   has $1024 \times 8 = 8192$ transitions. With minibatch size 64, each
   epoch runs $8192/64 = 128$ gradient steps. Four epochs = **512 gradient
   steps per rollout**. Compare to A2C's 1. This is the key to PPO's sample
   efficiency — not just multiple epochs, but the combination of large rollouts
   with small minibatches. (Hyperparameters from rl-baselines3-zoo.)

3. **Clipped surrogate loss** — the core of PPO.  Instead of the standard
   policy gradient $\log\pi(a|s)\cdot\hat{A}$, we use the probability ratio
   $r_t(\theta) = \pi_\theta/\pi_{\theta_{\text{old}}}$ clipped to prevent
   large steps.

Everything else — network, optimiser, GAE, gradient clipping — is unchanged.


In [ ]:
def ppo_clip(
    actor, critic, opt,
    n_envs:        int   = 8,
    n_steps:       int   = 1024,   # long rollouts → rich GAE estimates
    total_steps:   int   = 1_000_000,
    n_epochs:      int   = 4,      # reuse each rollout 4 times
    minibatch_size:int   = 64,     # small batches → many gradient steps per rollout
    gamma:         float = 0.999,  # SB3-Zoo tuned for LunarLander
    lam:           float = 0.98,   # SB3-Zoo tuned for LunarLander
    clip_coef:     float = 0.2,    # ε — the trust-region width
    entropy_coef:  float = 0.01,
    value_coef:    float = 0.5,
    max_grad_norm: float = 0.5,
    print_every:   int   = 100_000,
    seed:          int   = 42,
) -> tuple[list, list]:
    """
    PPO-Clip on LunarLander-v3.

    Rollout loop (same as A2C, no_grad):
      Collect n_steps transitions from n_envs parallel environments.

    Update loop (new in PPO):
      for epoch in range(n_epochs):            # reuse the rollout K times
          for minibatch in shuffled_rollout:   # minibatch gradient step
              ratio    = exp(new_log_prob - old_log_prob)
              pg_loss1 = -advantage * ratio
              pg_loss2 = -advantage * clip(ratio, 1-ε, 1+ε)
              actor_loss = mean(max(pg_loss1, pg_loss2))   ← pessimistic lower bound
              critic_loss = 0.5 * MSE(new_value, return)
              total_loss  = actor_loss + value_coef*critic_loss - entropy_coef*H
              backward + clip_grad + step

    Hyperparameters match rl-baselines3-zoo tuned config for LunarLander-v3.
    """
    envs = gym.vector.SyncVectorEnv(
        [lambda: gym.make("LunarLander-v3") for _ in range(n_envs)]
    )
    obs, _ = envs.reset(seed=seed)
    obs = torch.as_tensor(obs, dtype=torch.float32, device=device)

    ep_buf = np.zeros(n_envs)
    all_ep_rew, step_at_ep = [], []
    batch_size = n_steps * n_envs   # total transitions per rollout: 1024×8 = 8192

    # Shape legend (with defaults): T=n_steps=1024, N=n_envs=8,
    #   S=STATE_DIM=8, A=ACTION_DIM=4, B=batch_size=8192, M=minibatch_size=64
    n_updates = total_steps // batch_size

    for update in range(n_updates):
        steps_so_far = update * batch_size

        # ── 1. Allocate rollout buffers ───────────────────────────────────────
        obs_b  = torch.zeros(n_steps, n_envs, STATE_DIM, device=device)        # (T, N, S) = (1024, 8, 8)
        act_b  = torch.zeros(n_steps, n_envs, dtype=torch.long, device=device) # (T, N)    = (1024, 8)
        lp_b   = torch.zeros(n_steps, n_envs, device=device)   # old log-probs # (T, N)    = (1024, 8)
        rew_b  = torch.zeros(n_steps, n_envs, device=device)                   # (T, N)    = (1024, 8)
        don_b  = torch.zeros(n_steps, n_envs, device=device)                   # (T, N)    = (1024, 8)
        val_b  = torch.zeros(n_steps, n_envs, device=device)                   # (T, N)    = (1024, 8)

        # ── 2. Collect rollout (no_grad — these are the "old" policy values) ──
        for step in range(n_steps):
            with torch.no_grad():
                action, log_prob, _ = actor.get_action(obs)  # action,log_prob: (N,) = (8,)
                value = critic(obs)                            # value:           (N,) = (8,)

            next_obs, reward, terminated, truncated, _ = envs.step(action.cpu().numpy())
            done = (terminated | truncated).astype(np.float32)  # reward, done: (N,) = (8,)

            ep_buf += reward
            for i, d in enumerate(done):
                if d:
                    all_ep_rew.append(float(ep_buf[i]))
                    step_at_ep.append(steps_so_far + (step + 1) * n_envs)
                    ep_buf[i] = 0.0

            obs_b[step] = obs;   act_b[step] = action   # write row step: (N,S) and (N,)
            lp_b[step]  = log_prob
            rew_b[step] = torch.as_tensor(reward, dtype=torch.float32, device=device)
            don_b[step] = torch.as_tensor(done,   dtype=torch.float32, device=device)
            val_b[step] = value

            obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)  # (N, S) = (8, 8)

        # ── 3. Bootstrap + GAE ────────────────────────────────────────────────
        with torch.no_grad():
            next_val = critic(obs)   # V(s_{T+1}):  (N,) = (8,)

        advantages, returns = compute_gae(
            rew_b, val_b.detach(), don_b, next_val, gamma, lam
        )
        # advantages: (B,) = (8192,) normalised
        # returns:    (B,) = (8192,) critic regression targets

        # ── 4. Flatten buffers: (T, N, ...) → (T*N, ...) = (B, ...) ───────────
        obs_flat = obs_b.reshape(-1, STATE_DIM)   # (B, S) = (8192, 8)
        act_flat = act_b.reshape(-1)               # (B,)   = (8192,)
        lp_old   = lp_b.reshape(-1).detach()       # (B,)   = (8192,)  frozen old log-probs

        # ── 5. PPO update: K epochs × minibatches ────────────────────────────
        for epoch in range(n_epochs):
            # Shuffle all transitions so minibatches are diverse
            perm = torch.randperm(batch_size, device=device)   # (B,) = (8192,)

            for start in range(0, batch_size, minibatch_size):
                mb_idx = perm[start : start + minibatch_size]   # (M,) = (64,)

                # Re-evaluate current policy on this minibatch
                # obs_flat[mb_idx]: (M, S) = (64, 8);  act_flat[mb_idx]: (M,) = (64,)
                _, new_log_probs, entropy = actor.get_action(obs_flat[mb_idx], act_flat[mb_idx])
                new_values = critic(obs_flat[mb_idx])   # new_log_probs, entropy, new_values: (M,) = (64,)

                # Probability ratio  r_t(θ) = π_θ(a|s) / π_θ_old(a|s)
                # Use log-space for numerical stability: exp(log π_new - log π_old)
                ratio = torch.exp(new_log_probs - lp_old[mb_idx])   # (M,) = (64,)

                mb_adv = advantages[mb_idx]   # (M,) = (64,)

                # ── Clipped surrogate loss ─────────────────────────────────
                # Unclipped: r_t * A_t
                pg_loss1 = -mb_adv * ratio                                          # (M,) = (64,)
                # Clipped:   clip(r_t, 1-ε, 1+ε) * A_t
                pg_loss2 = -mb_adv * torch.clamp(ratio, 1 - clip_coef, 1 + clip_coef)  # (M,) = (64,)
                # Pessimistic lower bound: take the worse (max of negatives)
                # This kills the gradient whenever ratio is outside [1-ε, 1+ε]
                actor_loss = torch.max(pg_loss1, pg_loss2).mean()   # scalar ()

                # ── Critic loss ────────────────────────────────────────────
                critic_loss = value_coef * F.mse_loss(new_values, returns[mb_idx].detach())  # scalar ()

                # ── Entropy bonus ──────────────────────────────────────────
                entropy_loss = -entropy_coef * entropy.mean()   # scalar ()

                total_loss = actor_loss + critic_loss + entropy_loss   # scalar ()

                opt.zero_grad()
                total_loss.backward()
                nn.utils.clip_grad_norm_(
                    list(actor.parameters()) + list(critic.parameters()), max_grad_norm)
                opt.step()

        steps_done = (update + 1) * batch_size
        if steps_done // print_every > (steps_done - batch_size) // print_every and all_ep_rew:
            avg = np.mean(all_ep_rew[-100:]) if len(all_ep_rew) >= 100 else np.mean(all_ep_rew)
            solved = "✓ SOLVED" if avg >= 200 else ""
            print(f"[PPO]  step {steps_done:7d}  eps={len(all_ep_rew):4d}  avg100: {avg:7.1f}  {solved}")

    envs.close()
    return all_ep_rew, step_at_ep


---
## 7b  Aside — How the Minibatch Indexing Works (`obs_flat[mb_idx]`)

The PPO update above leans on one PyTorch operation that's easy to gloss over:
`obs_flat[mb_idx]`.  It's worth understanding exactly, because it's how every
minibatch is assembled.

### Basic indexing first

With a simple 1D tensor:

```python
x = torch.tensor([10, 20, 30, 40, 50])
#  index:           0   1   2   3   4

x[2]        # → tensor(30)          single integer: get one element
x[1:4]      # → tensor([20, 30, 40]) slice: get a contiguous range, in order
```

### Fancy indexing — pass a *list/tensor* of indices

```python
idx = torch.tensor([3, 0, 4])   # pick indices 3, 0, 4 — in that order
x[idx]                          # → tensor([40, 10, 50])
```

Three things to notice:

- the result has the same length as `idx`, **not** the length of `x`;
- the order follows `idx` — index 0 comes *second* here, because it's second in `idx`;
- indices can be non-consecutive and in any order.

### Now the 2D case (the actual code)

`obs_flat` has shape `(8192, 8)` — 8192 observations, each with 8 features:

```
obs_flat:
  row 0:    [x0, x1, x2, x3, x4, x5, x6, x7]   ← observation 0
  row 1:    [x0, x1, x2, x3, x4, x5, x6, x7]   ← observation 1
  ...
  row 8191: [...]
```

`mb_idx` has shape `(64,)` — 64 row numbers drawn from the shuffled `perm`:

```python
mb_idx = tensor([3, 7, 1, 5, 0, ...])   # 64 indices from 0..8191
```

When you write `obs_flat[mb_idx]`, PyTorch reads the index tensor as *"which
rows do I want?"*, picks exactly those 64 rows, and stacks them — result shape
`(64, 8)`:

```
obs_flat[mb_idx]:
  row 0: obs_flat[3] = [x0, ..., x7]   ← was row 3
  row 1: obs_flat[7] = [x0, ..., x7]   ← was row 7
  row 2: obs_flat[1] = [x0, ..., x7]   ← was row 1
  ...
```

The same `mb_idx` applied to `act_flat[mb_idx]`, `lp_old[mb_idx]`,
`advantages[mb_idx]` picks the *corresponding* rows — so observation row 3 is
paired with action row 3, old-log-prob row 3, advantage row 3.  Everything
stays aligned because they all index with the same tensor.

### Why not just slice `obs_flat[start:start+64]`?

Slicing always picks **consecutive rows in original order**:

```python
obs_flat[0:64]    # rows 0..63   — all from the start
obs_flat[64:128]  # rows 64..127 — the next chunk
```

The problem: consecutive rows in the rollout buffer are consecutive *time
steps from the same environments* — highly correlated.  A minibatch of steps
0–63 sees the same 8 envs at the same point in their trajectories.

With the shuffled `perm`, `mb_idx = perm[start:start+64]` gives 64 indices
scattered across all 8 environments and all 1024 time steps — diverse and
decorrelated.  The gradient from that minibatch is a much better estimate of
the true gradient.

### The whole thing in two lines

```python
perm   = torch.randperm(8192)   # shuffle all 8192 row indices
mb_idx = perm[0:64]             # first 64 shuffled indices, e.g. [3071, 44, 7823, ...]

obs_flat[mb_idx]    # (64, 8)  — 64 randomly chosen observations
act_flat[mb_idx]    # (64,)    — their corresponding actions
advantages[mb_idx]  # (64,)    — their advantages
```

All three index with the same `mb_idx`, so row $i$ of each always describes the
**same transition**.


In [ ]:
# Run this to see fancy indexing live ──────────────────────────────────────────
x = torch.tensor([10, 20, 30, 40, 50])
print("x          =", x)
print("x[2]       =", x[2], "  (single element)")
print("x[1:4]     =", x[1:4], "(slice, in order)")

idx = torch.tensor([3, 0, 4])
print("idx        =", idx)
print("x[idx]     =", x[idx], "  (fancy: same length & order as idx)")

# 2D: pick rows out of a (5, 2) matrix
mat = torch.arange(10).reshape(5, 2)
rows = torch.tensor([3, 0, 4])
print("\nmat =\n", mat)
print("mat[rows] =\n", mat[rows], "  ← rows 3, 0, 4 stacked → shape", tuple(mat[rows].shape))


---
## 8  Train — Same Budget, Same Seed

Both algorithms receive the same 300k steps for A2C and 500k for PPO.
PPO gets more steps because it's the algorithm we're showcasing — and it
needs to convincingly cross 200 to make the point.

Hyperparameters from [rl-baselines3-zoo](https://github.com/DLR-RM/rl-baselines3-zoo)
tuned config for LunarLander-v3 (γ=0.999, λ=0.98 — higher than Acrobot
because landing requires long-horizon credit assignment).

**Expected runtime on Colab CPU:** ~20 minutes total (LunarLander box2d is heavier than Acrobot).
GPU does not help here — the bottleneck is the box2d physics stepping on CPU.


In [ ]:
SEED       = 42
HIDDEN_DIM = 256
LR         = 3e-4    # SB3 PPO default

torch.manual_seed(SEED)
np.random.seed(SEED)

def make_actor_critic(seed):
    torch.manual_seed(seed)
    actor  = ActorNetwork(STATE_DIM, ACTION_DIM, HIDDEN_DIM).to(device)
    critic = CriticNetwork(STATE_DIM, HIDDEN_DIM).to(device)
    return actor, critic


In [ ]:
# ── Part A: A2C ───────────────────────────────────────────────────────────────
actor_a, critic_a = make_actor_critic(SEED)
opt_a_actor  = optim.Adam(actor_a.parameters(),  lr=LR, eps=1e-5)
opt_a_critic = optim.Adam(critic_a.parameters(), lr=LR, eps=1e-5)

print("=" * 60)
print("Part A — A2C + GAE  (500k steps · n_steps=5 · γ=0.995)")
print("=" * 60)
rewards_a2c, steps_a2c = a2c_gae(
    actor_a, critic_a, opt_a_actor, opt_a_critic,
    n_envs=8, n_steps=5, total_steps=500_000,
    gamma=0.995, lam=0.95, seed=SEED,
)
print(f"\nA2C  final avg (last 100): {np.mean(rewards_a2c[-100:]):.1f}")
print(f"     solved (avg ≥ 200)    {'✓ SOLVED' if np.mean(rewards_a2c[-100:]) >= 200 else 'not solved'}")


In [ ]:
# ── Part B: PPO ───────────────────────────────────────────────────────────────
actor_b, critic_b = make_actor_critic(SEED)
# PPO uses a single combined optimiser — both networks share one Adam instance
opt_ppo = optim.Adam(
    list(actor_b.parameters()) + list(critic_b.parameters()),
    lr=LR, eps=1e-5
)

print("=" * 60)
print("Part B — PPO-Clip  (1M steps · 4 epochs · mb=64 · ε=0.2)")
print("=" * 60)
rewards_ppo, steps_ppo = ppo_clip(
    actor_b, critic_b, opt_ppo,
    n_envs=8, n_steps=1024, total_steps=1_000_000, seed=SEED,
)
print(f"\nPPO  final avg (last 100): {np.mean(rewards_ppo[-100:]):.1f}")
print(f"     solved (avg ≥ 200)    {'✓ SOLVED' if np.mean(rewards_ppo[-100:]) >= 200 else 'not solved'}")


---
## 9  Comparison — A2C vs PPO


In [ ]:
def smooth(steps, rewards, window=50):
    """Trailing-window smoothed curve, returns (steps, smoothed)."""
    out = []
    for i in range(len(rewards)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(rewards[lo : i + 1])))
    return steps, out


fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(steps_a2c, rewards_a2c, alpha=0.1, color="steelblue", linewidth=0.7)
ax.plot(*smooth(steps_a2c, rewards_a2c),
        color="steelblue", linewidth=2.5,
        label=f"A2C + GAE  (n_steps=5, γ=0.995, {len(rewards_a2c)} episodes)")

ax.plot(steps_ppo, rewards_ppo, alpha=0.1, color="darkorange", linewidth=0.7)
ax.plot(*smooth(steps_ppo, rewards_ppo),
        color="darkorange", linewidth=2.5,
        label=f"PPO-Clip   (4 epochs · mb=64, {len(rewards_ppo)} episodes)")

ax.axhline(200, color="red", linestyle="--", linewidth=1.5,
           alpha=0.8, label="Solved threshold (200)")

ax.set_xlabel("Total environment steps (same-scale comparison)", fontsize=12)
ax.set_ylabel("Episode reward", fontsize=12)
ax.set_title("A2C vs PPO-Clip on LunarLander-v3", fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("comparison.png", dpi=130)
plt.show()
print("Saved → comparison.png")


### Interpreting the results

**A2C (blue):** runs with its SB3-tuned hyperparameters (`n_steps=5`,
`γ=0.995`), giving 12,500 gradient updates in 500k steps.  Despite frequent
updates, it plateaus well below 200.  Each update is one unconstrained
gradient step — unlucky batches can push the policy backward, and with no
step-size limit there is no recovery mechanism.

**PPO (orange):** climbs more steadily and crosses 200.  The key difference:
every 8192-transition rollout yields 4 epochs × 128 minibatches = 512 gradient
steps, all safely bounded by the clip.  The pessimistic lower bound means
a single bad minibatch can hurt at most by $\varepsilon = 0.2$ in probability
space before the gradient dies.

#### Why the same data used 16 times is safe

After the first gradient step on a minibatch, the current policy $\pi_\theta$
differs from $\pi_{\theta_{\text{old}}}$.  On step 2, the ratio $r_t(\theta)$
is no longer 1.0.  If the policy drifted far from the rollout distribution,
$r_t$ exits $[1-\varepsilon, 1+\varepsilon]$ and the gradient stops.
The clip structurally prevents overfitting to stale data.

#### Seed and hardware variability

As in previous units, exact numbers depend on seed and device.  The
qualitative story — PPO solving LunarLander while A2C does not — is robust.


In [ ]:
def evaluate_greedy(actor: ActorNetwork, n_episodes: int = 100) -> float:
    eval_env = gym.make("LunarLander-v3")
    rewards = []
    actor.eval()
    with torch.no_grad():
        for _ in range(n_episodes):
            state, _ = eval_env.reset()
            total, done = 0.0, False
            while not done:
                s_t    = torch.as_tensor(state, dtype=torch.float32, device=device)
                action = actor(s_t).argmax().item()
                state, r, terminated, truncated, _ = eval_env.step(action)
                done = terminated or truncated
                total += r
            rewards.append(total)
    eval_env.close()
    actor.train()
    return float(np.mean(rewards))


score_a2c = evaluate_greedy(actor_a)
score_ppo = evaluate_greedy(actor_b)

print("Greedy evaluation (100 episodes, argmax policy):")
print(f"  A2C + GAE  : {score_a2c:7.1f}  {'✓ SOLVED' if score_a2c >= 200 else 'not solved'}")
print(f"  PPO-Clip   : {score_ppo:7.1f}  {'✓ SOLVED' if score_ppo >= 200 else 'not solved'}")


---
## 10  Watch the PPO Agent Land


In [ ]:
def record_gif(actor: ActorNetwork, filename: str = "agent.gif",
               max_steps: int = 1000) -> str:
    env_rec = gym.make("LunarLander-v3", render_mode="rgb_array")
    frames, total_reward = [], 0.0
    state, _ = env_rec.reset()
    done = False
    actor.eval()
    with torch.no_grad():
        while not done and len(frames) < max_steps:
            frames.append(env_rec.render())
            s_t    = torch.as_tensor(state, dtype=torch.float32, device=device)
            action = actor(s_t).argmax().item()
            state, r, terminated, truncated, _ = env_rec.step(action)
            done = terminated or truncated
            total_reward += r
    env_rec.close()
    actor.train()
    iio.imwrite(filename, np.stack(frames), plugin="pillow", loop=0, duration=33)
    print(f"{len(frames)} frames  |  total reward: {total_reward:.1f}  →  {filename}")
    return filename


gif_path = record_gif(actor_b)
display(Image(gif_path))


---
## 11  What's Next — RLHF

PPO is the workhorse of modern deep RL — it trains locomotion policies in
robotics, game-playing agents, and (with modifications) large language models.

In Units 1–3 we assumed the **reward function was given** by the environment.
In practice, for language models, there is no natural reward signal.
You cannot write a function that scores whether a paragraph is helpful or
harmful.

**RLHF** (Reinforcement Learning from Human Feedback, Ziegler et al. 2019)
solves this by learning a reward model from human preference data, then
fine-tuning a language model with PPO against that learned reward.

The policy gradient update is:

$$\theta \leftarrow \theta + \alpha\,\nabla_\theta
\mathbb{E}_{a \sim \pi_\theta}\!\left[r_\phi(x, a)
- \beta\,\text{KL}\bigl(\pi_\theta(\cdot|x)\|\pi_{\text{ref}}(\cdot|x)\bigr)\right]$$

where $r_\phi$ is the learned reward model and $\pi_{\text{ref}}$ is the
original SFT model acting as a KL anchor.

Everything from Units 1–3 — policy gradient, advantage estimation, PPO's
clipped surrogate — carries forward directly.  The only new piece is the
reward model training.

→ [Back to the series](https://github.com/AliBuildsAI/rl-for-robotics-llms)
